In [1]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
from itertools import count

from node_b import NodeB
from multi_gnb_wrapper import MultiGNBWrapper

from slice_l1 import SliceL1eMBB
from slice_ran import SliceRANeMBB
from channel_models import SINRSelectiveFading, MCSCodeset
from schedulers import ProportionalFair
from traffic_generators import CbrSource, VbrSource

In [2]:
rng = default_rng(42)

slots_per_step = 50
slot_length = 1e-3
n_prbs = 100

CBR_description = {
    'lambda': 2.0 / 60.0,
    't_mean': 30.0,
    'bit_rate': 500000
}

VBR_description = {
    'lambda': 5.0 / 60.0,
    't_mean': 30.0,
    'p_size': 1000,
    'b_size': 500,
    'b_rate': 1
}

SLA_embb = {
    'cbr_th': 10e6,
    'cbr_prb': 20,
    'cbr_queue': 10e4,
    'vbr_th': 15e6,
    'vbr_prb': 30,
    'vbr_queue': 15e4
}

state_variables_embb = [
    'cbr_traffic', 'cbr_th', 'cbr_prb', 'cbr_queue', 'cbr_snr',
    'vbr_traffic', 'vbr_th', 'vbr_prb', 'vbr_queue', 'vbr_snr'
]

time_per_step = slots_per_step * slot_length
norm_const_embb = {
    'cbr_traffic': 5e6 * time_per_step,
    'cbr_th': 10e6 * time_per_step,
    'cbr_prb': 25 * slots_per_step,
    'cbr_queue': 10e4 * slots_per_step,
    'cbr_snr': 35 * slots_per_step,
    'vbr_traffic': 5e6 * time_per_step,
    'vbr_th': 10e6 * time_per_step,
    'vbr_prb': 35 * slots_per_step,
    'vbr_queue': 10e4 * slots_per_step,
    'vbr_snr': 35 * slots_per_step
}

In [3]:
def build_gnb(gnb_id, x, y, carrier_id, n_prbs=100, coverage_radius=300):
    snr_generator = SINRSelectiveFading(rng, 'macro_cell_urban_2GHz', n_prbs=n_prbs)
    mcs_codeset = MCSCodeset()
    scheduler = ProportionalFair(mcs_codeset)
    user_counter = count(gnb_id * 10000)

    slice_ran = SliceRANeMBB(
        rng=rng,
        user_counter=user_counter,
        id=0,
        SLA=SLA_embb,
        CBR_description=CBR_description,
        VBR_description=VBR_description,
        state_variables=state_variables_embb,
        norm_const=norm_const_embb,
        slots_per_step=slots_per_step,
        slot_length=slot_length
    )

    slice_l1 = SliceL1eMBB(
        rng=rng,
        snr_generator=snr_generator,
        n_prbs=n_prbs,
        slices_ran=[slice_ran],
        scheduler=scheduler
    )

    gnb = NodeB(
        id=gnb_id,
        x=x,
        y=y,
        slices_l1=[slice_l1],
        slots_per_step=slots_per_step,
        n_prbs=n_prbs,
        coverage_radius=coverage_radius,
        slot_length=slot_length,
        carrier_id=carrier_id,
        center_frequency_hz=3.5e9,
        bandwidth_hz=20e6,
        tx_power_dbm=30.0,
        noise_figure_db=7.0
    )
    return gnb

In [4]:
gnb1 = build_gnb(gnb_id=0, x=100, y=100, carrier_id=0, coverage_radius=280)
gnb2 = build_gnb(gnb_id=1, x=320, y=150, carrier_id=1, coverage_radius=280)

gnb_list = [gnb1, gnb2]

for gnb in gnb_list:
    print(gnb)

NodeB 0 at (100.00, 100.00), radius=280, carrier_id=0, f=3.50GHz, bw=20.0MHz
NodeB 1 at (320.00, 150.00), radius=280, carrier_id=1, f=3.50GHz, bw=20.0MHz


In [6]:
env = MultiGNBWrapper(
    gnb_list=gnb_list,
    handover_hysteresis=1.0,   # in dB
    handover_ttt=2,
    outage_penalty=1.0,
    handover_penalty=0.1,
    verbose=True
)

obs, info = env.reset()

print("Observation shape:", obs.shape)
print("Info:", info)
print("Action dimension:", env.action_space.shape)

Observation shape: (23,)
Info: {'step_count': 0, 'n_gnbs': 2, 'n_tracked_ues': 0, 'n_connected_ues': 0, 'n_disconnected_ues': 0, 'ue_per_gnb': [0, 0], 'per_gnb_rewards': [0.0, 0.0], 'mean_gnb_reward': 0.0, 'current_control_ue_id': None}
Action dimension: ()


In [7]:
ue1 = env.add_ue(x=120, y=120, vx=2.0, vy=0.5, slice_type="eMBB")
ue2 = env.add_ue(x=240, y=130, vx=1.0, vy=0.0, slice_type="eMBB")
ue3 = env.add_ue(x=300, y=160, vx=-1.0, vy=0.0, slice_type="eMBB")

print("UE IDs:", ue1, ue2, ue3)

for ue_id, ue in env.get_all_ues().items():
    print(ue_id, ue, ue.x, ue.y, ue.serving_gnb)

UE IDs: 0 1 2


AttributeError: 'list' object has no attribute 'items'

In [ ]:
for ue_id in env.get_all_ues():
    print(f"\nUE {ue_id}")
    print(env.get_ue_radio_metrics(ue_id))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['lightblue', 'lightgreen']

for i, gnb in enumerate(gnb_list):
    gnb.visualize_coverage(
        ax=ax,
        color=colors[i],
        alpha=0.25,
        edge_color='black',
        linewidth=1
    )
    ax.text(gnb.x, gnb.y, f"gNB{i}", ha='center', va='center', fontsize=10, fontweight='bold')

for ue_id, ue in env.get_all_ues().items():
    c = 'red' if ue.serving_gnb is None else 'blue'
    ax.plot(ue.x, ue.y, 'o', color=c, markersize=7)
    ax.text(ue.x + 5, ue.y + 5, f"UE{ue_id}", fontsize=9)

all_x = []
all_y = []
for gnb in gnb_list:
    all_x.extend([v[0] for v in gnb.vertices])
    all_y.extend([v[1] for v in gnb.vertices])

margin = 60
ax.set_xlim(min(all_x) - margin, max(all_x) + margin)
ax.set_ylim(min(all_y) - margin, max(all_y) + margin)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title("Initial layout: gNodeBs and UEs")
plt.show()

In [ ]:
reward_history = []

for t in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    reward_history.append(reward)

    print(f"\nStep {t}")
    print("Reward:", reward)
    print("Connected UEs:", info["n_connected_ues"])
    print("Disconnected UEs:", info["n_disconnected_ues"])
    print("Handover count:", info["handover_count_total"])

    for ue_id in env.get_all_ues():
        print(env.get_ue_radio_metrics(ue_id))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = ['lightblue', 'lightgreen']

for i, gnb in enumerate(gnb_list):
    gnb.visualize_coverage(
        ax=ax,
        color=colors[i],
        alpha=0.25,
        edge_color='black',
        linewidth=1
    )
    ax.text(gnb.x, gnb.y, f"gNB{i}", ha='center', va='center', fontsize=10, fontweight='bold')

for ue_id, ue in env.get_all_ues().items():
    c = 'red' if ue.serving_gnb is None else 'blue'
    ax.plot(ue.x, ue.y, 'o', color=c, markersize=7)
    ax.text(ue.x + 5, ue.y + 5, f"UE{ue_id}->g{ue.serving_gnb}", fontsize=9)

all_x = []
all_y = []
for gnb in gnb_list:
    all_x.extend([v[0] for v in gnb.vertices])
    all_y.extend([v[1] for v in gnb.vertices])

margin = 60
ax.set_xlim(min(all_x) - margin, max(all_x) + margin)
ax.set_ylim(min(all_y) - margin, max(all_y) + margin)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title("After stepping the wrapper")
plt.show()

In [ ]:
trajectories = {ue_id: [(ue.x, ue.y)] for ue_id, ue in env.get_all_ues().items()}

for t in range(50):
    action = env.action_space.sample()
    env.step(action)

    for ue_id, ue in env.get_all_ues().items():
        trajectories[ue_id].append((ue.x, ue.y))
fig, ax = plt.subplots(figsize=(8,6))

# plot gNBs
for gnb in env.gnbs:
    gnb.visualize_coverage(ax=ax, alpha=0.2)

# plot trajectories
for ue_id, traj in trajectories.items():
    traj = np.array(traj)
    ax.plot(traj[:,0], traj[:,1], '-o', label=f'UE{ue_id}')

ax.set_title("UE trajectories")
ax.legend()
ax.grid()
plt.show()


In [ ]:
gnb1.carrier_id = 0
gnb2.carrier_id = 0
env._update_all_ue_radio_states()

print("Different carriers")
for ue_id in env.get_all_ues():
    print(env.get_ue_radio_metrics(ue_id))

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(reward_history, marker='o')
plt.title("Reward history")
plt.xlabel("Step")
plt.ylabel("Reward")
plt.grid(True, alpha=0.3)
plt.show()

UE mobility stress test with visible handovers + SINR drop zones

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
from itertools import count

from node_b import NodeB
from multi_gnb_wrapper import MultiGNBWrapper

from slice_l1 import SliceL1eMBB
from slice_ran import SliceRANeMBB
from channel_models import SINRSelectiveFading, MCSCodeset
from schedulers import ProportionalFair

In [ ]:
rng = default_rng(42)

slots_per_step = 50
slot_length = 1e-3
n_prbs = 100

CBR_description = {
    'lambda': 2.0 / 60.0,
    't_mean': 30.0,
    'bit_rate': 500000
}

VBR_description = {
    'lambda': 5.0 / 60.0,
    't_mean': 30.0,
    'p_size': 1000,
    'b_size': 500,
    'b_rate': 1
}

SLA_embb = {
    'cbr_th': 10e6,
    'cbr_prb': 20,
    'cbr_queue': 10e4,
    'vbr_th': 15e6,
    'vbr_prb': 30,
    'vbr_queue': 15e4
}

state_variables_embb = [
    'cbr_traffic', 'cbr_th', 'cbr_prb', 'cbr_queue', 'cbr_snr',
    'vbr_traffic', 'vbr_th', 'vbr_prb', 'vbr_queue', 'vbr_snr'
]

time_per_step = slots_per_step * slot_length
norm_const_embb = {
    'cbr_traffic': 5e6 * time_per_step,
    'cbr_th': 10e6 * time_per_step,
    'cbr_prb': 25 * slots_per_step,
    'cbr_queue': 10e4 * slots_per_step,
    'cbr_snr': 35 * slots_per_step,
    'vbr_traffic': 5e6 * time_per_step,
    'vbr_th': 10e6 * time_per_step,
    'vbr_prb': 35 * slots_per_step,
    'vbr_queue': 10e4 * slots_per_step,
    'vbr_snr': 35 * slots_per_step
}

In [ ]:
def build_gnb(gnb_id, x, y, carrier_id, n_prbs=100, coverage_radius=280):
    snr_generator = SINRSelectiveFading(rng, 'macro_cell_urban_2GHz', n_prbs=n_prbs)
    mcs_codeset = MCSCodeset()
    scheduler = ProportionalFair(mcs_codeset)
    user_counter = count(gnb_id * 10000)

    slice_ran = SliceRANeMBB(
        rng=rng,
        user_counter=user_counter,
        id=0,
        SLA=SLA_embb,
        CBR_description=CBR_description,
        VBR_description=VBR_description,
        state_variables=state_variables_embb,
        norm_const=norm_const_embb,
        slots_per_step=slots_per_step,
        slot_length=slot_length
    )

    slice_l1 = SliceL1eMBB(
        rng=rng,
        snr_generator=snr_generator,
        n_prbs=n_prbs,
        slices_ran=[slice_ran],
        scheduler=scheduler
    )

    return NodeB(
        id=gnb_id,
        x=x,
        y=y,
        slices_l1=[slice_l1],
        slots_per_step=slots_per_step,
        n_prbs=n_prbs,
        coverage_radius=coverage_radius,
        slot_length=slot_length,
        carrier_id=carrier_id,
        center_frequency_hz=3.5e9,
        bandwidth_hz=20e6,
        tx_power_dbm=30.0,
        noise_figure_db=7.0
    )

In [ ]:
gnb1 = build_gnb(0, 100, 100, carrier_id=0, coverage_radius=280)
gnb2 = build_gnb(1, 360, 130, carrier_id=0, coverage_radius=280)

env = MultiGNBWrapper(
    gnb_list=[gnb1, gnb2],
    handover_hysteresis=1.0,
    handover_ttt=2,
    verbose=False
)

obs, info = env.reset()
print(info)

In [ ]:
ue_ids = []

# crosses from gNB1 toward gNB2
ue_ids.append(env.add_ue(x=40, y=110, vx=25.0, vy=0.0, slice_type="eMBB"))

# moves diagonally through overlap
ue_ids.append(env.add_ue(x=120, y=40, vx=18.0, vy=10.0, slice_type="eMBB"))

# starts near overlap and moves across it
ue_ids.append(env.add_ue(x=220, y=120, vx=15.0, vy=0.0, slice_type="eMBB"))

# starts inside gNB2 and moves left
ue_ids.append(env.add_ue(x=500, y=150, vx=-20.0, vy=0.0, slice_type="eMBB"))

for ue_id in ue_ids:
    print(env.get_ue_radio_metrics(ue_id))

In [ ]:
ue_ids = []

# crosses from gNB1 toward gNB2
ue_ids.append(env.add_ue(x=40, y=110, vx=25.0, vy=0.0, slice_type="eMBB"))

# moves diagonally through overlap
ue_ids.append(env.add_ue(x=120, y=40, vx=18.0, vy=10.0, slice_type="eMBB"))

# starts near overlap and moves across it
ue_ids.append(env.add_ue(x=220, y=120, vx=15.0, vy=0.0, slice_type="eMBB"))

# starts inside gNB2 and moves left
ue_ids.append(env.add_ue(x=500, y=150, vx=-20.0, vy=0.0, slice_type="eMBB"))

for ue_id in ue_ids:
    print(env.get_ue_radio_metrics(ue_id))

In [ ]:
n_steps = 120

trajectories = {ue_id: [] for ue_id in ue_ids}
serving_history = {ue_id: [] for ue_id in ue_ids}
sinr_history = {ue_id: [] for ue_id in ue_ids}
handover_steps = []

for t in range(n_steps):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)

    if info["handover_count_step"] > 0:
        handover_steps.append(t)

    for ue_id in ue_ids:
        ue = env.get_ue(ue_id)
        trajectories[ue_id].append((ue.x, ue.y))
        serving_history[ue_id].append(ue.serving_gnb)
        sinr_history[ue_id].append(ue.sinr)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

gnb1.visualize_coverage(ax=ax, color='lightblue', alpha=0.20, edge_color='blue', linewidth=2)
gnb2.visualize_coverage(ax=ax, color='lightgreen', alpha=0.20, edge_color='green', linewidth=2)

colors = ['red', 'orange', 'purple', 'brown']

for i, ue_id in enumerate(ue_ids):
    traj = np.array(trajectories[ue_id])
    ax.plot(traj[:, 0], traj[:, 1], '-', color=colors[i], linewidth=2, label=f'UE{ue_id}')
    ax.plot(traj[0, 0], traj[0, 1], 'o', color=colors[i], markersize=8)
    ax.plot(traj[-1, 0], traj[-1, 1], 's', color=colors[i], markersize=7)

    # mark handover points where serving gNB changes
    s_hist = serving_history[ue_id]
    for t in range(1, len(s_hist)):
        if s_hist[t] != s_hist[t-1]:
            xh, yh = traj[t]
            ax.plot(xh, yh, 'kx', markersize=10, markeredgewidth=2)

all_x = [v[0] for v in gnb1.vertices] + [v[0] for v in gnb2.vertices]
all_y = [v[1] for v in gnb1.vertices] + [v[1] for v in gnb2.vertices]

margin = 80
ax.set_xlim(min(all_x) - margin, max(all_x) + margin)
ax.set_ylim(min(all_y) - margin, max(all_y) + margin)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title("UE mobility stress test with visible handovers")
ax.legend()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for ue_id in ue_ids:
    plt.plot(sinr_history[ue_id], label=f'UE{ue_id}')

plt.axhline(20, linestyle='--')
plt.axhline(10, linestyle='--')
plt.axhline(0, linestyle='--')

plt.title("SINR over time")
plt.xlabel("Step")
plt.ylabel("SINR (dB)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(ue_ids), 1, figsize=(10, 2.2 * len(ue_ids)), sharex=True)

if len(ue_ids) == 1:
    axes = [axes]

for ax, ue_id in zip(axes, ue_ids):
    ax.plot(serving_history[ue_id], drawstyle='steps-post')
    ax.set_ylabel(f'UE{ue_id}')
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Step")
fig.suptitle("Serving gNB history")
plt.show()

In [ ]:
for ue_id in ue_ids:
    print(f"\nUE {ue_id}")
    print(env.get_ue_radio_metrics(ue_id))